In [1]:
import os
import json
from datetime import datetime, timezone
from pathlib import Path

from pyspark.context import SparkContext
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField,
    LongType, IntegerType, DoubleType, TimestampType, StringType, DateType
)

Configuration, constant paths

In [2]:
BASE_DIR = Path()
INBOX_DIR = BASE_DIR / "data" / "inbox"
OUTBOX_DIR = BASE_DIR / "data" / "outbox"
STATE_DIR = BASE_DIR / "state"
OUTPUT_PATH = OUTBOX_DIR / "trips_enriched.parquet"
LOOKUP_PATH = BASE_DIR / "data" / "taxi_zone_lookup.parquet"
MANIFEST_PATH = STATE_DIR / "manifest.json"

State management

In [3]:
def load_manifest() -> dict:
    """Load state from disk or return empty one"""
    if MANIFEST_PATH.exists():
        with open(MANIFEST_PATH) as file:
            manifest = json.load(file)
            if "processed_files" in manifest.keys():
                return manifest
            #return json.load(file)
    return {"processed_files": {}}

def save_manifest(manifest: dict) -> None:
    """Save state to disk"""
    STATE_DIR.mkdir(parents=True, exist_ok=True)
    tmp = MANIFEST_PATH.with_suffix(".tmp")
    with open(tmp, "w") as file:
        json.dump(manifest, file, indent=2, default=str)
    tmp.replace(MANIFEST_PATH)
    print("Manifest saved to", MANIFEST_PATH)

def file_fingerprint(path: Path) -> dict:
    stat = path.stat()
    return {
        "filename": path.name,
        "filepath": str(path),
        "size_bytes": stat.st_size,
        "mtime": stat.st_mtime
    }

def new_files(inbox: Path, manifest: dict) -> list[Path]:
    processed = set(manifest["processed_files"].keys())
    candidates = sorted(inbox.glob("*.parquet"))
    fresh = [p for p in candidates if p.name not in processed]
    print(f"Found {len(candidates)} parquet files in inbox, {len(fresh)} are new")
    return fresh


Ingestion

In [4]:
def read_trip_files(spark: SparkSession, paths: list[Path]) -> DataFrame:
    frames = []
    for p in paths:
        df = spark.read.parquet(str(p))
        df = df.withColumn("source_file", F.lit(p.name))
        frames.append(df)

    if not frames:
        raise ValueError("Paths list is empty")
    
    result = frames[0]
    for df in frames[1:]:
        result = result.unionByName(df, allowMissingColumns=True)

    print("Raw schema:", result.schema.simpleString())
    return result

def read_zone_lookup(spark: SparkSession) -> DataFrame:
    return spark.read.parquet(str(LOOKUP_PATH))

Casting and normalization, cleaning etc.

Main entry

In [5]:
def main():
    
    INBOX_DIR.mkdir(parents=True, exist_ok=True)
    OUTBOX_DIR.mkdir(parents=True, exist_ok=True)
    STATE_DIR.mkdir(parents=True, exist_ok=True)

    manifest = load_manifest()
    print(f"Manifest loaded, {len(manifest["processed_files"])} files previously processed")

    files_to_process = new_files(INBOX_DIR, manifest)

    if not files_to_process:
        print("No new files found, exiting")
        return
    
    ingested_at = datetime.now(timezone.utc).isoformat()

    sc = SparkContext('local', 'project_01')
    spark = (
        SparkSession.builder
        .appName('project_01')
        .enableHiveSupport()
        .getOrCreate()
    )
    print(spark.version)

    try:
        # Ingest
        raw_df = read_trip_files(spark, files_to_process)
        zones = read_zone_lookup(spark)

        # Cast types
        df = raw_df

        # Clean

        # Deduplicate

        # Enrich with zones

        # Assemble final schema

        # Merge with existing output
        full_df = df
        total_rows = full_df.count()

        # Write output

        # Update manifest
        new_row_count = df.count()
        for p in files_to_process:
            fp = file_fingerprint(p)
            fp["row_count"] = new_row_count
            fp["processed_at"] = ingested_at
            manifest["processed_files"][p.name] = fp

        manifest["last_run"] = ingested_at
        manifest["total_output_rows"] = total_rows
        save_manifest(manifest)

    finally:
        spark.stop()
        

In [6]:
main()

Manifest loaded, 0 files previously processed
Found 2 parquet files in inbox, 2 are new
4.1.0
Raw schema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passenger_count:bigint,trip_distance:double,RatecodeID:bigint,store_and_fwd_flag:string,PULocationID:int,DOLocationID:int,payment_type:bigint,fare_amount:double,extra:double,mta_tax:double,tip_amount:double,tolls_amount:double,improvement_surcharge:double,total_amount:double,congestion_surcharge:double,Airport_fee:double,cbd_congestion_fee:double,source_file:string>
Manifest saved to state/manifest.json
